# Coverage rollup and column contract input

Reads every profile emitted by notebooks 01 to 05 and produces two artefacts:

1. **A consolidated findings table** — every check, its verdict and its detail, which
   is the evidence behind the Tuesday coverage gate.
2. **A draft known-limitation register and column contract note list**, which feed
   DMP section 10.3 and the Stage 3 transform respectively.

This notebook derives nothing new. If a figure is not in a profile, it does not appear
here.

In [1]:
import sys, warnings
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
warnings.filterwarnings("ignore")

import pandas as pd
import profile_lib as pl

pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

print("project root:", pl.PROJECT_ROOT)
print("raw zone:    ", pl.RAW_ROOT)


project root: C:\Users\nitin\Documents\Projects\Final_Project\SportAble
raw zone:     C:\Users\nitin\Documents\Projects\Final_Project\SportAble\_raw


In [2]:
profiles = pl.load_profiles()
print(f"{len(profiles)} profiles loaded: {sorted(profiles)}")
meta = pd.DataFrame([
    {
        "dataset": k,
        "title": v["title"],
        "dt": v["dt"],
        "object": v["source_object"],
        "size_mb": round(v["size_bytes"] / 1e6, 2),
        "sha256_short": v["sha256"][:12],
        "manifest_hash": {True: "match", False: "MISMATCH", None: "not recorded"}[v["sha_matches_manifest"]],
        "verdict": v["worst_verdict"].upper(),
    }
    for k, v in sorted(profiles.items())
])
meta

7 profiles loaded: ['DS-01', 'DS-02', 'DS-03', 'DS-04', 'DS-06', 'DS-07', 'DS-08']


,dataset,title,dt,object,size_mb,sha256_short,manifest_hash,verdict
0,DS-01,Sport and Recreational Facilities List,2026-08-31,srv_ifmd_all-facilities.xlsx,3.66,d7a3992688ab,match,FAIL
1,DS-02,National Public Toilet Map,2026-08-31,toiletmapexport_260801_074429.csv,12.00,8aaef33e53f0,match,WARN
2,DS-03,PTV GTFS Schedule,2026-08-31,gtfs.zip,288.40,d492e2bc8def,match,WARN
3,DS-04,"Accessible Parking Locations, City of Melbourne",2026-08-31,CityofMelbourneaccessibledisabilityparking.kml,0.75,5a7ea935ee36,match,WARN
4,DS-06,"ABS LGA boundaries 2025, GDA2020",2026-08-31,LGA_2025_AUST_GDA2020.zip,40.73,acc3015a0ac7,match,PASS
5,DS-07,"ABS Suburbs and Localities 2021, GDA2020",2026-08-31,SAL_2021_AUST_GDA2020_SHP.zip,104.06,1284f6aa4a5e,match,PASS
6,DS-08,"ABS Suburbs and Localities 2021, GDA2020",2026-08-31,SAL_2021_AUST_GDA2020_SHP.zip,104.06,1284f6aa4a5e,match,PASS


In [3]:
rows = []
for ds, v in sorted(profiles.items()):
    for c in v["checks"]:
        rows.append({"dataset": ds, "check": c["key"], "verdict": c["verdict"].upper(), "detail": c["detail"]})
findings = pd.DataFrame(rows)
order = {"FAIL": 0, "WARN": 1, "INFO": 2, "PASS": 3}
findings = findings.sort_values(["verdict", "dataset"], key=lambda s: s.map(order).fillna(9))
findings

,dataset,check,verdict,detail
6,DS-01,coords_in_range,FAIL,7 rows have a coordinate outside valid lat/lon...
1,DS-01,grain,WARN,"9,590 rows resolve to 5,000 facilities — the s..."
2,DS-01,dedup_collapse_rules,WARN,6 venue-level columns take more than one value...
5,DS-01,coords_present,WARN,"354 of 9,590 rows have a null coordinate — the..."
7,DS-01,coords_not_transposed,WARN,5 rows are invalid as given but valid when tra...
10,DS-01,truncation_Facility Features,WARN,"Facility Features is cut at 255 characters, 1,..."
11,DS-01,facility_features_fragments,WARN,71 of 89 tokens in Facility Features are prefi...
12,DS-01,tristate_required,WARN,accessible toilet and accessible parking from ...
21,DS-02,boolean_semantics,WARN,19 accessibility booleans are never null in th...
27,DS-03,feed_info_present,WARN,"8 mode feeds ship no feed_info.txt, so the fee..."


In [4]:
counts = findings["verdict"].value_counts()
blocking = findings[findings["verdict"] == "FAIL"]
print(counts.to_string())
print()
if len(blocking):
    print("BLOCKING — these must be resolved before the Stage 3 transform is written:")
    for _, r in blocking.iterrows():
        print(f"  {r['dataset']}  {r['check']}: {r['detail']}")
else:
    print("No blocking failures. The column contracts can be written.")

verdict
PASS    36
INFO    15
WARN    14
FAIL     1

BLOCKING — these must be resolved before the Stage 3 transform is written:
  DS-01  coords_in_range: 7 rows have a coordinate outside valid lat/lon range


## Draft known-limitation register

DMP section 10.3 generates the AC2.1.5 limits text from this register, so an entry added here is an entry that appears on the venue card.

In [5]:
lim = [{"dataset": ds, "limitation": t} for ds, v in sorted(profiles.items()) for t in v["limitations"]]
lim_df = pd.DataFrame(lim)
for _, r in lim_df.iterrows():
    print(f"[{r['dataset']}] {r['limitation']}\n")
lim_df

[DS-01] DS-01 uses superseded council names for Greater Dandenong, Merri-bek. The rename is applied from a recorded alias table, not inferred by fuzzy matching.

[DS-01] DS-01 Facility Features is truncated at 255 characters. Presence of an accessible toilet or accessible parking token is evidence; absence is not. Rows at maximum field length are reported as no published information.

[DS-01] DS-01 records changeroom presence and gender but never changeroom accessibility. The change link of the access chain is not answerable from this source alone.

[DS-02] DS-02 publishes accessibility attributes as booleans with no separate unknown value. A false is treated as no published information rather than as a surveyed absence.

[DS-03] DS-03 records no wheelchair boarding information for 87.0% of Greater Melbourne stops. Those stops are shown as no published information, not as inaccessible.

[DS-04] DS-04 publishes no identifier, so the primary key is derived from the coordinate and street 

,dataset,limitation
0,DS-01,DS-01 uses superseded council names for Greate...
1,DS-01,DS-01 Facility Features is truncated at 255 ch...
2,DS-01,DS-01 records changeroom presence and gender b...
3,DS-02,DS-02 publishes accessibility attributes as bo...
4,DS-03,DS-03 records no wheelchair boarding informati...
5,DS-04,"DS-04 publishes no identifier, so the primary ..."
6,DS-04,"DS-04 covers the City of Melbourne only, one o..."
7,DS-04,The publisher states that disability parking l...
8,DS-04,"A point is a bay, not a route to a bay. The la..."
9,DS-04,An on-street bay is not venue parking. Off-str...


## Column contract notes

Every rule the Stage 3 transform has to implement, gathered from the profiles.

In [6]:
con = [{"dataset": ds, "rule": t} for ds, v in sorted(profiles.items()) for t in v["contract_notes"]]
con_df = pd.DataFrame(con)
for _, r in con_df.iterrows():
    print(f"[{r['dataset']}] {r['rule']}\n")
con_df

[DS-01] Load sheet 'wholeIFMD' only. Reject the file if that sheet name is absent.

[DS-01] Deduplicate on Facility ID before load. Sports Played becomes a child table or a collapsed array, never a duplicated venue row.

[DS-01] Write an explicit collapse rule per column listed in columns_varying_within_facility. Do not rely on drop_duplicates ordering.

[DS-01] Clip to Greater Melbourne spatially against DS-06 before load. Do not filter on LGA name in the API.

[DS-01] Rows with a null or out-of-range coordinate are quarantined with reason code COORD_MISSING. They are never geocoded.

[DS-01] Derive DS-01 access tokens as tri-state. Absence of a token in a row at maximum field length is NOT_PUBLISHED, never false.

[DS-02] Clip DS-02 spatially against DS-06 rather than filtering on State or Town.

[DS-02] FacilityID is the natural key for DS-02 and is enforced unique on load.

[DS-02] Map DS-02 booleans to tri-state: true -> PUBLISHED_YES, false -> NOT_RECORDED, null -> NOT_RECORDED. 

,dataset,rule
0,DS-01,Load sheet 'wholeIFMD' only. Reject the file i...
1,DS-01,Deduplicate on Facility ID before load. Sports...
2,DS-01,Write an explicit collapse rule per column lis...
3,DS-01,Clip to Greater Melbourne spatially against DS...
4,DS-01,Rows with a null or out-of-range coordinate ar...
5,DS-01,Derive DS-01 access tokens as tri-state. Absen...
6,DS-02,Clip DS-02 spatially against DS-06 rather than...
7,DS-02,FacilityID is the natural key for DS-02 and is...
8,DS-02,Map DS-02 booleans to tri-state: true -> PUBLI...
9,DS-02,"Carry KeyRequired, MLAK24 and PaymentRequired ..."


## Sources absent from this assessment

The register holds seven sources; six appear above. DS-05 (openrouteservice) is a live
routing API with `tier: live`. It lands no object in the raw zone by design, so it has
no Stage 2 profile and no SHA-256 to profile against. Its coverage question is a quota
and terms question handled in the source register, not a data profiling question.

The per-LGA coverage table required by DMP section 10.2 is not produced here. It needs
the spatial clip against the LGA boundary layer, which is a Stage 3 output. This
assessment covers what can be established from the landed objects alone.

Two sources were registered earlier in Iteration 1 and withdrawn on 31 August 2026: the
City of Melbourne on-street car park bay restrictions and on-street parking bays. They
could not be joined and are replaced by DS-04. The evidence is retained in their retired
source cards.


## Write the coverage report

In [7]:
dt = meta["dt"].iloc[0]
out_dir = pl.PROJECT_ROOT / "_profiles" / f"dt={dt}"
lines = [
    f"# SportAble Melbourne — Stage 2 coverage assessment",
    f"",
    f"Partition `dt={dt}`. Generated from the profile records emitted by notebooks 01-05.",
    f"Every figure below is traceable to a named raw object and its SHA-256.",
    f"",
    "## Objects profiled",
    "",
    meta.to_markdown(index=False),
    "",
    "## Findings",
    "",
    findings.to_markdown(index=False),
    "",
    "## Known-limitation register (draft)",
    "",
    (lim_df.to_markdown(index=False) if len(lim_df) else "_none recorded_"),
    "",
    "## Sources absent from this assessment",
    "",
    "The register holds seven sources. DS-05 (openrouteservice) is a live routing API, tier live, "
    "which lands no object in the raw zone by design and therefore has no Stage 2 profile. The per-LGA "
    "coverage table required by DMP section 10.2 needs the spatial clip and is a Stage 3 output.",
    "",
    "## Column contract rules for Stage 3",
    "",
    (con_df.to_markdown(index=False) if len(con_df) else "_none recorded_"),
    "",
]
report = out_dir / "coverage_assessment.md"
report.write_text("\n".join(lines), encoding="utf-8")
print("Written:", report)

Written: C:\Users\nitin\Documents\Projects\Final_Project\SportAble\_profiles\dt=2026-08-31\coverage_assessment.md
